In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer

import joblib
import os

In [2]:
# Словарь для преобразования русских названий целей в английские
GOAL_MAP = {
    'похудение': 'weight_loss',
    'снижение веса': 'weight_loss',
    'похудеть': 'weight_loss',
    'набор мышц': 'muscle_gain',
    'массанабор': 'muscle_gain',
    'набор': 'muscle_gain',
    'набрать массу': 'muscle_gain',
    'накачаться': 'muscle_gain',
    'поддержание': 'maintenance',
    'поддержать вес': 'maintenance',
    'низкоуглеводная': 'low_carb',
    'низкоуглеводная диета': 'low_carb',
    'без углеводов': 'low_carb',
    'кетодиета': 'low_carb',
    'кето': 'low_carb',
}

# Словарь для преобразования русских названий типов блюд в английские
MEAL_TYPE_RU_TO_EN = {
    'завтрак': 'breakfast',
    'обед': 'lunch',
    'ужин': 'dinner',
    'десерт': 'dessert',
    'закуска': 'snack',
    'перекус': 'snack',
    'суп': 'lunch',
    'салат': 'snack',
    'выпечка': 'dessert',
    'основное блюдо': 'dinner',
    'напиток': 'other',
    'соус': 'other',
    'заготовка': 'other',
    'бульон': 'lunch',
    'паста': 'dinner',
    'пицца': 'dinner',
    'рис': 'dinner'
}

# Обратный словарь для отображения английских типов на русские
MEAL_TYPE_EN_TO_RU = {
    'breakfast': 'завтрак',
    'lunch': 'обед',
    'dinner': 'ужин',
    'dessert': 'десерт',
    'snack': 'закуска',
    'other': 'другое'
}

def map_dish_type_to_meal_type(dish_type):

    if not isinstance(dish_type, str):
        return 'other'
    
    dish_lower = dish_type.lower()
    
    if 'завтрак' in dish_lower or 'breakfast' in dish_lower:
        return 'breakfast'

    elif 'суп' in dish_lower or 'бульон' in dish_lower or 'борщ' in dish_lower or 'окрошк' in dish_lower or 'обед' in dish_lower:
        return 'lunch'

    elif 'основное блюдо' in dish_lower or 'мясо' in dish_lower or 'рыб' in dish_lower or 'куриц' in dish_lower:
        return 'dinner'
    
    elif 'десерт' in dish_lower or 'выпечк' in dish_lower or 'пирог' in dish_lower or 'торт' in dish_lower or 'печень' in dish_lower:
        return 'dessert'
    
    elif 'закуск' in dish_lower or 'салат' in dish_lower or 'бутерброд' in dish_lower or 'сэндвич' in dish_lower:
        return 'snack'
    
    elif 'паст' in dish_lower or 'пицц' in dish_lower:
        return 'dinner'
    
    elif 'напиток' in dish_lower:
        return 'other'
    
    elif 'заготовк' in dish_lower:
        return 'other'
    
    elif 'соус' in dish_lower:
        return 'other'
    
    else:
        return 'other'


In [2]:
recipes = pd.read_parquet("../data/processed/rus_recipes_clean_final.parquet")

In [3]:
rf_data = recipes.copy()
rf_data['ingredients_str'] = rf_data['ingredients_list'].apply(
        lambda x: ' '.join(x) if isinstance(x, list) else str(x)
    )
if 'meal_type' not in rf_data.columns and 'dish_type' in rf_data.columns:
    meal_map = {
        'Завтрак': 'breakfast',
        'Суп': 'lunch',
        'Бульон': 'lunch',
        'Основное блюдо': 'dinner',
        'Выпечка': 'dessert',
        'Закуска': 'snack',
        'Салат': 'snack',
        'Напиток': 'other',
        'Соус': 'other',
        'Заготовка': 'other',
        'Паста или пицца': 'dinner',
        'Сэндвич': 'snack',
        'Ризотто': 'dinner',
    }
    rf_data['meal_type'] = rf_data['dish_type'].map(meal_map).fillna('other')

In [4]:
# создание целевых переменных для каждой цели пользователя

def calculate_suitability_score(row, goal):
    protein = row['protein_g']
    fat = row['fat_g']
    carbs = row['carbs_g']
    calories = row['calories']
    
    # вычисляем проценты
    total = protein + fat + carbs
    if total == 0:
        return 0
    
    protein_pct = protein / total * 100
    fat_pct = fat / total * 100
    carbs_pct = carbs / total * 100
    
    if goal == 'weight_loss':
        score = 0
        if protein_pct >= 40:
            score += 0.4
        elif protein_pct >= 30:
            score += 0.3
        elif protein_pct >= 20:
            score += 0.2
        if fat_pct <= 25:
            score += 0.3
        elif fat_pct <= 35:
            score += 0.2
        elif fat_pct <= 45:
            score += 0.1
        if 20 <= carbs_pct <= 40:
            score += 0.3
        elif carbs_pct < 50:
            score += 0.2
        if calories < 400:
            score += 0.1
        elif calories < 600:
            score += 0.05
        
        return score
    
    elif goal == 'muscle_gain':
        score = 0
        if protein_pct >= 35:
            score += 0.5
        elif protein_pct >= 25:
            score += 0.3
        if 20 <= fat_pct <= 35:
            score += 0.25
        elif 15 <= fat_pct <= 40:
            score += 0.15
        if 20 <= carbs_pct <= 40:
            score += 0.25
        
        return score
    
    elif goal == 'maintenance':
        score = 0
  
        if 25 <= protein_pct <= 35:
            score += 0.40
        elif 20 <= protein_pct <= 40:
            score += 0.25
        elif 18 <= protein_pct <= 45:
            score += 0.10
        if 25 <= fat_pct <= 35:
            score += 0.35
        elif 20 <= fat_pct <= 40:
            score += 0.25
        elif 15 <= fat_pct <= 45:
            score += 0.10
        if 30 <= carbs_pct <= 40:
            score += 0.25
        elif 25 <= carbs_pct <= 45:
            score += 0.20
        elif 20 <= carbs_pct <= 50:
            score += 0.10
        if 400 <= calories <= 600:
            score += 0.10
        elif 300 <= calories <= 700:
            score += 0.05
        
        return score
    
    elif goal == 'low_carb':
        score = 0
        if carbs_pct <= 20:
            score += 0.5
        elif carbs_pct <= 30:
            score += 0.3
        if protein_pct >= 35:
            score += 0.3
        elif protein_pct >= 25:
            score += 0.2
        if 30 <= fat_pct <= 50:
            score += 0.2
        
        return score
    
    return 0


In [ ]:

for goal in ['weight_loss', 'muscle_gain', 'maintenance', 'low_carb']:
    col_name = f'suitable_{goal}'

    scores = rf_data.apply(lambda row: calculate_suitability_score(row, goal), axis=1)

    threshold = np.percentile(scores, 60) 
    rf_data[col_name] = (scores >= threshold).astype(int)
    
# индексы для разделения
indices = np.arange(len(rf_data))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.2, 
    random_state=0, 
    stratify=rf_data['suitable_weight_loss'] 
)

# разделяем данные
train_data = rf_data.iloc[train_idx].copy()
test_data = rf_data.iloc[test_idx].copy()


In [ ]:
tfidf = TfidfVectorizer(max_features=100) 
train_tfidf = tfidf.fit_transform(train_data['ingredients_str'])
test_tfidf = tfidf.transform(test_data['ingredients_str'])


train_tfidf_df = pd.DataFrame(
    train_tfidf.toarray(),
    columns=[f'ing_{i}' for i in range(train_tfidf.shape[1])],
    index=train_data.index
)

test_tfidf_df = pd.DataFrame(
    test_tfidf.toarray(),
    columns=[f'ing_{i}' for i in range(test_tfidf.shape[1])],
    index=test_data.index
)

In [7]:
numeric_features = ['calories', 'protein_g', 'fat_g', 'carbs_g']
train_meal_dummies = pd.get_dummies(train_data['meal_type'], prefix='meal')
test_meal_dummies = pd.get_dummies(test_data['meal_type'], prefix='meal')

In [8]:
for col in train_meal_dummies.columns:
    if col not in test_meal_dummies.columns:
        test_meal_dummies[col] = 0


test_meal_dummies = test_meal_dummies[train_meal_dummies.columns]


X_train = pd.concat([
    train_data[numeric_features],
    train_meal_dummies,
    train_tfidf_df
], axis=1)

X_test = pd.concat([
    test_data[numeric_features],
    test_meal_dummies,
    test_tfidf_df
], axis=1)

In [ ]:
# подбор параметров
# param_grid= {
#     'n_estimators': [200, 300],
#     'max_depth': [15, 20, 25],
#     'min_samples_split': [2, 5],
# }
# goals = {
#     'weight_loss': 'suitable_weight_loss',
#     'muscle_gain': 'suitable_muscle_gain',
#     'maintenance': 'suitable_maintenance',
#     'low_carb': 'suitable_low_carb'
# }
# from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# best_models = {}
# best_params = {}
# cv_results = {}

# for goal_name, target_col in goals.items():
  
#     y_train = train_data[target_col]
#     y_test = test_data[target_col]
    
#     rf = RandomForestClassifier(random_state=42, n_jobs=-1)
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 
#     grid_search = GridSearchCV(
#         estimator=rf,
#         param_grid=param_grid,
#         cv=cv,
#         scoring='f1',
#         n_jobs=-1,
#         verbose=1,
#         return_train_score=True
#     )
#     grid_search.fit(X_train, y_train)
    
#     best_models[goal_name] = grid_search.best_estimator_
#     best_params[goal_name] = grid_search.best_params_
#     cv_results[goal_name] = grid_search.cv_results_
    
#     print(f"\n лучшие параметры для {goal_name}:")
#     for param, value in grid_search.best_params_.items():
#         print(f"  {param}: {value}")
    
#     print(f"\n лучший F1 (CV): {grid_search.best_score_:.4f}")
    

#     y_test_pred = grid_search.best_estimator_.predict(X_test)
#     y_train_pred = grid_search.best_estimator_.predict(X_train)
    
#     train_f1 = f1_score(y_train, y_train_pred)
#     test_f1 = f1_score(y_test, y_test_pred)
#     train_precision = precision_score(y_train, y_train_pred)
#     test_precision = precision_score(y_test, y_test_pred)
#     train_recall = recall_score(y_train, y_train_pred)
#     test_recall = recall_score(y_test, y_test_pred)
    
#     print(f"\n метрики на тесте:")
#     print(f"  F1:        {test_f1:.4f}")
#     print(f"  Precision: {test_precision:.4f}")
#     print(f"  Recall:    {test_recall:.4f}")
    


In [ ]:
# обучение моделей
models = {}

goals = {
    'weight_loss': 'suitable_weight_loss',
    'muscle_gain': 'suitable_muscle_gain',
    'maintenance': 'suitable_maintenance',
    'low_carb': 'suitable_low_carb'
}

print("\nПараметры:")
print("Количество деревьев: 300")
print("Максимальная глубина: 25")
print("Минимальное число объектов в узле: 2")

for goal_name, target_col in goals.items():
    print(f"{'_'*60}")
    print(f"\nОбучение модели для {goal_name}")
    # Берем target для train и test
    y_train = train_data[target_col]
    y_test = test_data[target_col]
    
    
    # Обучаем модель
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=25,
        min_samples_split=2,
        random_state=0,
        class_weight='balanced',
    )
    
    
    rf.fit(X_train, y_train)
    models[goal_name] = rf 

    y_train_pred = rf.predict(X_train)
    train_f1 = f1_score(y_train, y_train_pred)
    train_recall = recall_score(y_train, y_train_pred)
    train_precision = precision_score( y_train, y_train_pred )
    
    # Оцениваем качество на тесте
    y_test_pred = rf.predict(X_test)
    test_f1 = f1_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_precision = precision_score( y_test, y_test_pred )
    
    print(f"\nМетрики:")
    print(f"  Train F1: {train_f1:.4f}")
    print(f"  Test F1: {test_f1:.4f}")
    print(f"  Train Recall: { train_recall:.4f}")
    print(f"  Test Recall: {test_recall:.4f}")
    print(f"  Train Precision: {train_precision:.4f}")
    print(f"  Test Precision: {test_precision:.4f}")
joblib.dump(models, '../model/rf_models.pkl')




Параметры:
Количество деревьев: 300
Максимальная глубина: 25
Минимальное число объектов в узле: 2
____________________________________________________________

Обучение модели для weight_loss

Метрики:
  Train F1: 0.9991
  Test F1: 0.9182
  Train Recall: 0.9998
  Test Recall: 0.9166
  Train Precision: 0.9983
  Test Precision: 0.9198
____________________________________________________________

Обучение модели для muscle_gain

Метрики:
  Train F1: 0.9994
  Test F1: 0.8996
  Train Recall: 0.9989
  Test Recall: 0.9060
  Train Precision: 1.0000
  Test Precision: 0.8934
____________________________________________________________

Обучение модели для maintenance

Метрики:
  Train F1: 0.9965
  Test F1: 0.8532
  Train Recall: 0.9995
  Test Recall: 0.8713
  Train Precision: 0.9935
  Test Precision: 0.8358
____________________________________________________________

Обучение модели для low_carb

Метрики:
  Train F1: 0.9997
  Test F1: 0.9399
  Train Recall: 0.9994
  Test Recall: 0.9425
  Train

['../model/rf_models.pkl']